# Generating STAC metadata for DPS job outputs

**Authors:** Henry Rodman (Development Seed)

**Date:** January 6, 2026

**Description:** DPS job outputs are automatically uploaded to S3 with a prefix that is determined by your username, the algorithm name, job identifier, and the time at which the job finishes. To improve accessibility of DPS output files, STAC metadata that are written alongside your normal job outputs are automatically ingested into a STAC and made accessible via a STAC API ([dps-stac.maap-project.org](dps-stac.maap-project.org)) and a titiler API ([titiler-dps-stac.maap-project.org](titiler-dps-stac.maap-project.org)). This tutorial will explain how to produce STAC metadata as part of a DPS algorithm and how to use the APIs to access and visualize the outputs.

## How to write STAC metadata

The last step in your DPS algorithm script is usually to write the output files in a cloud-optimized format. To take advantage of the DPS STAC capability you will need to write some STAC metadata to describe the output files after they are created.

- algorithm that writes a COG
- describe importance of relative hrefs
- show how to do it when writing a file to output/file.tif
- create Catalog, Item, write to output/catalog.json

In [15]:
from datetime import datetime, UTC
from pathlib import Path

import rio_stac
import rioxarray
from pystac import Asset, Catalog, CatalogType, MediaType

This algorithm creates an xarray DataArray from an input COG and writes it to the output directory for the DPS job.

In [2]:
da = rioxarray.open_rasterio(
    "s3://maap-ops-workspace/henrydevseed/dps_output/HLSCloudFreeTemporalMosaic/v0.1.1/mic-check/2025/11/06/02/55/51/893808/red.tif"
)

# this is usually defined in your algorithm's run script
output_dir = Path("/tmp/output")
output_dir.mkdir(exist_ok=True)

output_file = "demo.tif"
da.rio.to_raster(
    output_dir / output_file,
    driver="COG",
)

### Generating a STAC item

There are many ways to create STAC item metadata, but the easiest way to get going is with [rio-stac](https://github.com/developmentseed/rio-stac). rio-stac can generate a boilerplate STAC item with only a path to an input raster file.

If you are generating vector files (e.g. geoparquet, geopackage, etc) you will need to use a different approach.

In [3]:
rio_stac.create_stac_item(
    output_dir / output_file
)

<Item id=demo.tif>

However we need to provide some additional arguments in order to make this STAC item useful. Most importantly the **item ID** which needs to be unique within this job's STAC collection. If we run multiple jobs that create STAC items with the same ID they will overwrite eachother in the STAC database.

Let's say that the algorithm has two input parameters that determine a unique job: spatial extent and year/month. I will generate the STAC item ID using those factors.

In [4]:
bbox_str = "_".join(str(int(x)) for x in da.rio.bounds())
item_id = f"2025-05__{bbox_str}"

print(item_id)

2025-05__-245760_2211840_0_2457600


Now create the item with the unique ID and some other enriching attributes. You can add anything you want to the `properties` dict! Use this to include attributes that you might want to use for filtering STAC items later.

Please use the `.validate()` method when generating STAC metadata because it will help you catch small STAC metadata errors early rather than causing failures during the STAC item ingestion process.

In [16]:
item = rio_stac.create_stac_item(
    output_dir / output_file,
    id=item_id,
    input_datetime=datetime(2025, 5, 1, tzinfo=UTC),
    asset_media_type=MediaType.COG,  # advertise that the file is a COG!
    asset_roles=["data"],
    with_proj=True,  # let rio-stac add info about the file's projection
    properties={
        "grade": "A+",  # you can add any relevant attributes/tags here!
    }
)
item.set_self_href(f"{output_dir}/item.json")
item.validate()
item

<Item id=2025-05__-245760_2211840_0_2457600>

If your algorithm produces more than one file you will (in most cases) want to add it as an additional asset in the same STAC item.

In [17]:
output_file_2 = "additional.tif"
da.rio.to_raster(
    output_dir / output_file_2,
    driver="COG",
)

additional_asset = Asset(
    href=output_dir / output_file_2,
    media_type=MediaType.COG,
    roles=["data"],
)

item.assets.update(
    {"extra": additional_asset}
)
item.validate()
item

<Item id=2025-05__-245760_2211840_0_2457600>

### Exporting the STAC metadata

Now that we have a STAC item that describes the job's output files we need to write it to the output directory that gets uploaded to S3. To get the STAC item to be detected by the DPS STAC ingestion process we simply need to create a STAC catalog.json file that references the item. During this step the most imporant thing is to make sure the asset `hrefs` are written as **relative** (not absolute) links. This will ensure that the links to the actual output files will still be valid after everything is uploaded to S3.

In [ ]:
catalog = Catalog(
    id="DPS",
    description="DPS output STAC items",
    catalog_type=CatalogType.SELF_CONTAINED,
)
catalog.set_self_href(f"{output_dir}/catalog.json")

# add the item to the catalog
catalog.add_item(item)

# IMPORTANT: ensure all hrefs are written as relative to the root
catalog.make_all_asset_hrefs_relative()

# run validate_all to check that the STAC metadata is valid
catalog.validate_all()

# write the STAC metadata (catalog.json and all items) to the output directory
catalog.normalize_and_save(root_href=str(output_dir))

The catalog object serves as a container for your STAC items. There is a process that listens for `catalog.json` files that get uploaded to the DPS output folders in S3 so this is the key to getting your STAC metadata loaded into the DPS STAC!

When you create the catalog using `pystac` and add the item to it, both will be written to the output folder when you run `catalog.normalize_and_save(root_href=str(output_dir))`.

In [22]:
catalog.to_dict()

{'type': 'Catalog',
 'id': 'DPS',
 'stac_version': '1.1.0',
 'description': 'DPS output STAC items',
 'links': [{'rel': 'root',
   'href': './catalog.json',
   'type': 'application/json'},
  {'rel': 'item',
   'href': './2025-05__-245760_2211840_0_2457600/2025-05__-245760_2211840_0_2457600.json',
   'type': 'application/geo+json'},
  {'rel': 'self',
   'href': '/tmp/output/catalog.json',
   'type': 'application/json'}]}

In [20]:
next(catalog.get_all_items())

<Item id=2025-05__-245760_2211840_0_2457600>

### Full example algorithm script

```python
import argparse
from datetime import datetime
from pathlib import Path

import rio_stac
import rioxarray
from pystac import Asset, Catalog, CatalogType, MediaType

def run(bbox: tuple[float, float, float, float], timestamp: datetime, output_dir: Path) -> None:

    # generate your data
    da = rioxarray.open_rasterio(
        "s3://maap-ops-workspace/henrydevseed/dps_output/HLSCloudFreeTemporalMosaic/v0.1.1/mic-check/2025/11/06/02/55/51/893808/red.tif"
    )

    # write a raster to the output directory
    output_dir.mkdir(exist_ok=True)
    output_file = "demo.tif"
    
    da.rio.to_raster(
        output_dir / output_file,
        driver="COG",
    )

    # generate the STAC item ID
    bbox_str = "_".join(str(int(x)) for x in da.rio.bounds())
    datetime_str = timestamp.strftime("%Y-%m-%d")
    item_id = f"{datetime_str}__{bbox_str}"

    # use rio_stac.create_stac_item to easily generate the STAC item
    item = rio_stac.create_stac_item(
        output_dir / output_file,
        id=item_id,
        input_datetime=datetime(2025, 5, 1, tzinfo=UTC),
        asset_media_type=MediaType.COG,  # advertise that the file is a COG!
        asset_roles=["data"],
        with_proj=True,  # let rio-stac add info about the file's projection
        properties={
            "grade": "A+",  # you can add any relevant attributes/tags here!
        }
    )

    # if you have other assets that you created as part of your job
    # add them to the item like this
    output_file_2 = "additional.tif"
    da.rio.to_raster(
        output_dir / output_file_2,
        driver="COG",
    )
    
    additional_asset = Asset(
        href=output_dir / output_file_2,
        description="another asset",
        media_type=MediaType.COG,
        roles=["data"],
    )

    # update the item's asset dict with the new asset
    item.assets.update(
        {"other": additional_asset}
    )

    # set the href for this item in the output directory
    item.set_self_href(f"{output_dir}/item.json")

    # IMPORTANT: validate the STAC item to avoid heartburn later
    item.validate()

    # create a STAC catalog to house your item
    catalog = Catalog(
        id="DPS",
        description="DPS output STAC items",
        catalog_type=CatalogType.SELF_CONTAINED,
    )
    catalog.set_self_href(f"{output_dir}/catalog.json")
    
    # add the item to the catalog
    catalog.add_item(item)
    
    # IMPORTANT: ensure all hrefs are written as relative to the root
    catalog.make_all_asset_hrefs_relative()
    
    # run validate_all to check that the STAC metadata is valid
    catalog.validate_all()
    
    # write the STAC metadata (catalog.json and all items) to the output directory
    catalog.normalize_and_save(root_href=str(output_dir))


if __name__ == "__main__":
    parse = argparse.ArgumentParser(
        description="Example algorithm that produces a STAC item for the job results"
    )
    parse.add_argument(
        "--bbox",
        help="bounding box (xmin, ymin, xmax, ymax)",
        required=True,
        nargs=4,
        type=float,
        metavar=("xmin", "ymin", "xmax", "ymax"),
    )
    parse.add_argument(
        "--timestamp",
        help="timestamp to use for the STAC item",
        required=True,
        type=datetime,
    )
    parse.add_argument(
        "--output_dir", help="Directory in which to save output", required=True
    )

    args = parse.parse_args()

    run(**vars(args))
```

## What happens next

When your DPS job runs successfully, the contents of the output directory will be uploaded to S3. There is a process that listens for `catalog.json` files that are uploaded to DPS output locations and processes the items that are included in them.

The items are posted to the DPS STAC and are associated with a STAC collection that is defined by several job input parameters:

- your MAAP username
- the algorithm name
- the algorithm version/tag
- the job identifier/tag

So, for example, if I (`henrydevseed`) submit several jobs to version `v0.1` of the `BestAlgorithm` with a job identifier `final`, the STAC items that are produced will be added to a collection with ID `henrydevseed__BestAlgorithm__v0.1__final`.

If you change any of those parameters (username, algorithm name, job identifier, version, etc), STAC items will be associated with a different collection!

In the future there will be a way for you to specify the collection ID directly but for now the collection ID will be determined by these job parameters.

The STAC metadata will be made available in a publicly accessible STAC API but access to the actual files does not change.

## How to use the STAC metadata

- search for files using pystac-client
- visualize using the titiler API